## Tokenizer

In [4]:
with open("TheMetamorphosis.txt", "r", encoding = "utf-8") as f:
    raw_text = f.read()

print("total chars: ", len(raw_text))
print(raw_text[:100])
    

total chars:  118421
One morning, when Gregor Samsa woke from troubled dreams, he found
himself transformed in his bed in


In [52]:
import re

data = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
data = [word.strip() for word in data if word.strip()]

print(data[:39])
print("total words: ", len(data))

['One', 'morning', ',', 'when', 'Gregor', 'Samsa', 'woke', 'from', 'troubled', 'dreams', ',', 'he', 'found', 'himself', 'transformed', 'in', 'his', 'bed', 'into', 'a', 'horrible', 'vermin', '.', 'He', 'lay', 'on', 'his', 'armour-like', 'back', ',', 'and', 'if', 'he', 'lifted', 'his', 'head', 'a', 'little', 'he']
total words:  24331


In [53]:
words = sorted(set(data))
vocab_size = len(words)

print(words[:30])
print(vocab_size)

['!', '(', ')', ',', '.', ':', ';', '?', 'A', 'Across', 'After', 'All', 'Although', 'An', 'And', 'Anna', 'Another', 'Anyway', 'Aren’t', 'Arms', 'As', 'At', 'Be', 'Because', 'Before', 'Behind', 'Besides', 'But', 'By', 'Can’t']
2952


In [54]:
vocab = {s:i for i,s in enumerate(words)}
print(len(vocab.items()))

2952


In [55]:
class Tokenizer:
    def __init__(self, vocab):

        self.stoi = vocab
        self.itos = {i:s for s,i in vocab.items()}

    def encode(self, text):
        data = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        data = [word.strip() for word in data if word.strip()]

        ids = [self.stoi[i] for i in data]
        return ids
    def decode(self, ids):
        text = " ".join(self.itos[i] for i in ids)
        text = re.sub(r's+([,.:;?_!"()\'])', r'\1', text)
        return text

In [56]:
t1 = Tokenizer(vocab)

text = "when Gregor Samsa woke from troubled dreams"
ids = t1.encode(text)
decoded_text = t1.decode(ids)

print("encoded ids:",ids)
print("decoded text:",decoded_text)

encoded ids: [2764, 53, 114, 2810, 1125, 2599, 845]
decoded text: when Gregor Samsa woke from troubled dreams


In [60]:
words.extend(["<|endoftext|>", "<|unk|>"])
vocab_size = len(words)

vocab = {s:i for i,s in enumerate(words)}

In [61]:
class TokenizerV2:
    def __init__(self, vocab):

        self.stoi = vocab
        self.itos = {i:s for s,i in vocab.items()}

    def encode(self, text):
        data = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        data = [word.strip() for word in data if word.strip()]

        data = [word if word in self.stoi else "<|unk|>" for word in data]
        
        ids = [self.stoi[i] for i in data]
        return ids

        
    def decode(self, ids):
        text = " ".join(self.itos[i] for i in ids)
        text = re.sub(r's+([,.:;?_!"()\'])', r'\1', text)
        return text

In [63]:
t2 = TokenizerV2(vocab)

text = "Fucking Gregor Samsa woke from troubled dreams"
ids = t2.encode(text)
decoded_text = t2.decode(ids)

print("encoded ids:",ids)
print("decoded text:",decoded_text)

encoded ids: [2959, 53, 114, 2810, 1125, 2599, 845]
decoded text: <|unk|> Gregor Samsa woke from troubled dreams


In [66]:
#! pip install tiktoken


In [67]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


In [68]:
tokenizer = tiktoken.get_encoding("gpt2")

In [69]:
text = ("Fucking Gregor Samsa woke from troubled dreams")

tIds = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
dText = tokenizer.decode(tIds)

print(tIds)
print(dText)

[37, 19296, 8547, 273, 3409, 11400, 19092, 422, 17840, 10625]
Fucking Gregor Samsa woke from troubled dreams


## DataLoader

In [74]:
#! pip install torch

In [88]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, context_len, stride):
        self.input_ids = []
        self.output_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - context_len, stride):
            input_chunk = token_ids[i: i + context_len]
            output_chunk = token_ids[i + context_len: i + context_len + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.output_ids.append(torch.tensor(output_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.output_ids[idx]
        
        

In [89]:
def GPTDataLoader(text, batch_size = 8, context_len = 4, stride = 4, shuffle = False, drop_last = True, num_workers = 0):

    dataset = GPTDataset(text, tokenizer, context_len, stride)

    dataloader = DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [93]:
dataloader = GPTDataLoader(raw_text, batch_size=8, context_len=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[ 3198,  3329,    11,   618],
        [ 8547,   273,  3409, 11400],
        [19092,   422, 17840, 10625],
        [   11,   339,  1043,   198],
        [38400,   944, 14434,   287],
        [  465,  3996,   656,   257],
        [12361,  3326,  1084,    13],
        [  679,  3830,   319,   465]])

Targets:
 tensor([[ 8547],
        [19092],
        [   11],
        [38400],
        [  465],
        [12361],
        [  679],
        [  198]])
